# 🏋️ SFT on Qwen2.5-7B-Instruct over Alpaca

This notebook is **domain-agnostic**. Set `DOMAIN_NAME` to whatever engagement you're working on (e.g. `asset_integrity`, `customer_grasps`, `ai_llm`, `legal_nda_review`, ...) and the notebook will create the YAML if it doesn't already exist, then train against it.

The template is auto-resolved to Qwen ChatML from the base-model ID.

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('.'))

# ── Pick your domain name for this engagement ──
DOMAIN_NAME = 'ai_llm'   # ← change to your engagement, e.g. 'asset_integrity', 'customer_grasps'
BASE_MODEL  = 'Qwen/Qwen2.5-7B-Instruct'

from app.config_loader import create_domain_config, domain_config_exists, load_domain_config

if not domain_config_exists(DOMAIN_NAME):
    create_domain_config(
        name=DOMAIN_NAME,
        system_prompt='You are a senior AI engineer specialising in LLM post-training, evaluation, and efficient inference.',
        constitution=[
            'Cite primary sources for any non-obvious claim.',
            'Prefer reproducible benchmarks (MMLU, GSM8K, MT-Bench) with explicit hardware.',
            'Never hallucinate paper titles, arXiv IDs, or numbers.',
        ],
    )
    print(f'✅ Created configs/domains/{DOMAIN_NAME}.yaml')
else:
    print(f'✔️  Using existing configs/domains/{DOMAIN_NAME}.yaml')

config = load_domain_config(DOMAIN_NAME)
print(config['domain_name'], '—', config['system_prompt'][:80])

In [ ]:
from app.trainers import AgnosticSFTTrainer

trainer = AgnosticSFTTrainer(
    config=config,
    base_model_id=BASE_MODEL,
    hf_dataset_config={
        'repo_id': 'tatsu-lab/alpaca',
        'split': 'train',
        'input_column': 'instruction',
        'output_column': 'output',
        'max_samples': 500,
    },
)
result = trainer.train()
result